In [ ]:
# ==========================================
# Exp-8: Image Generation using Variational Autoencoder (VAE)
# ==========================================

# Step 1: Import Required Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import time

# Step 2: Configure Parameters and Load Dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_size = 32
batch_size = 64
latent_dim = 64
num_epochs = 3
learning_rate = 1e-3

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is unavailable. Training will use CPU.")

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# CIFAR-10 avoids the CelebA Google Drive download issue
dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform
)

# Use a small subset for faster training
dataset = Subset(dataset, range(2000))

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

print("Number of images:", len(dataset))

# Step 3: Define Encoder Network
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(),

            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, start_dim=1)

        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        return mu, logvar

# Step 4: Define Decoder Network
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 128, 4, 4)
        return self.deconv(x)

# Step 5: Define VAE Model
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decoder(z)
        return reconstructed, mu, logvar

# Step 6: Define VAE Loss Function
def vae_loss(recon_x, x, mu, logvar):
    recon_loss = F.mse_loss(
        recon_x, x, reduction="sum"
    )

    kl_div = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return (recon_loss + kl_div) / x.size(0)

# Step 7: Create Model and Optimizer
model = VAE(latent_dim).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# Enable mixed precision on CUDA
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("VAE model created successfully!")

# Step 8: Train the VAE
loss_history = []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, _ in dataloader:
        images = images.to(
            device,
            non_blocking=(device.type == "cuda")
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            reconstructed, mu, logvar = model(images)
            loss = vae_loss(
                reconstructed, images, mu, logvar
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

    average_loss = total_loss / len(dataset)
    loss_history.append(average_loss)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Loss: {average_loss:.4f}"
    )

print(f"Training time: {time.time() - start_time:.1f} seconds")

# Step 9: Plot Training Loss
plt.figure(figsize=(7, 4))
plt.plot(
    range(1, num_epochs + 1),
    loss_history,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Loss per Image")
plt.title("VAE Training Loss")
plt.grid(True)
plt.show()

# Step 10: Generate New Images
model.eval()

with torch.no_grad():
    z = torch.randn(16, latent_dim, device=device)
    generated_images = model.decoder(z).cpu()

    # Convert images from [-1, 1] to [0, 1]
    generated_images = (generated_images * 0.5 + 0.5).clamp(0, 1)

# Step 11: Visualize Generated Images
grid = torchvision.utils.make_grid(
    generated_images,
    nrow=4
)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Images Generated using VAE")
plt.show()

Device: cpu
CUDA is unavailable. Training will use CPU.


 99%|█████████▉| 169M/170M [21:16<00:08, 147kB/s]